# 07 — Pre-ingestion quality gate and architecture decisions
**Goal:** Define pass/fail checks and decide when to route a document to better parsing.

**Must:** detect failed/empty extraction, preserve important identifiers, check source reference coverage, and record errors.
**Trade-off:** more checks increase latency but reduce silent corruption.

This is not retrieval evaluation yet; it tests content readiness before indexing.

In [ ]:
from pathlib import Path
import re
DATA = Path('../data') if Path('../data').exists() else Path('data')
source = (DATA / 'jes2_manual.md').read_text(encoding='utf-8')
important = ['$D SPOOL', '$D JOB', 'ABEND=S0C4']

def quality_report(text):
    return {
        'nonempty': bool(text.strip()),
        'chars': len(text),
        'required_term_coverage': sum(x in text for x in important) / len(important),
        'has_sections': bool(re.search(r'^#{1,6} ', text, re.M)),
        'email_count': len(re.findall(r'(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b', text))
    }

report = quality_report(source)
report

In [ ]:
assert report['nonempty']
assert report['required_term_coverage'] == 1.0
assert report['has_sections']
print('Baseline passed')

### Decision record template
For every experiment, write:
1. **Input:** which document class?
2. **Technique:** which parser/preprocessor/chunker?
3. **Observations:** what survived or broke?
4. **Trade-offs:** quality, latency, cost, data policy.
5. **Choice:** default strategy and conditions for fallback.

**Expected:** all baseline checks pass on the Markdown source. Coverage against a few known terms is a toy test; production needs representative labeled examples and robust quality metrics.

**Milestone exit:** you can explain and implement each of the eight pre-ingestion concerns, distinguish universal requirements from conditional processing, and identify a poor extraction before indexing.